# LangChain Model Integration

This notebook demonstrates how to connect LangChain to every LLM provider whose API key is present in the `.env` file:

| Provider | Env Key | LangChain Class |
|---|---|---|
| Anthropic (Claude) | `ANTHROPIC_API_KEY` | `ChatAnthropic` |
| OpenAI (GPT) | `OPENAI_API_KEY` | `ChatOpenAI` |
| xAI (Grok) | `GROK_API_KEY` | `ChatOpenAI` (OpenAI-compatible) |
| Google (Gemini) | `GOOGLE_API_KEY` | `ChatGoogleGenerativeAI` |

Each section loads the model and sends the same test message so the responses are easy to compare.

### Step 1 — Load Environment Variables

**What:** Reads the `.env` file once and pulls all four API keys into Python variables.

**Why:** All LangChain provider classes accept an `api_key` argument. Loading keys here in one place means every model section below can use them directly without re-reading the file. The `bool()` check confirms each key was found without printing the actual secret.

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY")
GROK_API_KEY      = os.getenv("GROK_API_KEY")
GOOGLE_API_KEY    = os.getenv("GOOGLE_API_KEY")

print("ANTHROPIC_API_KEY :", bool(ANTHROPIC_API_KEY))
print("OPENAI_API_KEY    :", bool(OPENAI_API_KEY))
print("GROK_API_KEY      :", bool(GROK_API_KEY))
print("GOOGLE_API_KEY    :", bool(GOOGLE_API_KEY))

ANTHROPIC_API_KEY : True
OPENAI_API_KEY    : True
GROK_API_KEY      : True
GOOGLE_API_KEY    : True


---
## 1. Anthropic — Claude

**What:** Connects to Anthropic's Claude models via `ChatAnthropic` from `langchain-anthropic`.

**Why `claude-haiku-4-5-20251001`:** Haiku is the fastest and cheapest Claude model — ideal for testing integrations. Swap the `model` string for `claude-sonnet-4-6` or `claude-opus-4-8` when you need higher reasoning quality.

**How `.invoke()` works:** It accepts a plain string (LangChain wraps it as a `HumanMessage` internally) and returns an `AIMessage` object. `.content` extracts the text from it.

In [4]:
from langchain_anthropic import ChatAnthropic

anthropic_llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    anthropic_api_key=ANTHROPIC_API_KEY,
)

response = anthropic_llm.invoke("What is LangChain? Answer in one sentence.")
print("Anthropic →", response.content)

Anthropic → LangChain is a framework for developing applications powered by large language models (LLMs) by providing tools to chain together LLM calls, manage prompts, and integrate with external data sources.


---
## 2. OpenAI — GPT

**What:** Connects to OpenAI's GPT models via `ChatOpenAI` from `langchain-openai`.

**Why `gpt-4o-mini`:** It is OpenAI's smallest and most cost-efficient multimodal model — good for quick tests. Replace with `gpt-4o` or `o3` for production-grade reasoning.

**How it differs from Anthropic:** The interface is identical (`ChatOpenAI` is the OpenAI equivalent of `ChatAnthropic`). LangChain's unified `ChatModel` abstraction means you only need to swap the class and model name — the rest of your pipeline stays the same.

In [ ]:
from langchain_openai import ChatOpenAI

openai_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
)

response = openai_llm.invoke("What is LangChain? Answer in one sentence.")
print("OpenAI →", response.content)

---
## 3. xAI — Grok

**What:** Connects to xAI's Grok model. Grok exposes an **OpenAI-compatible REST API**, so we reuse `ChatOpenAI` and point it at xAI's endpoint instead of OpenAI's.

**Why `base_url`:** By passing `base_url="https://api.x.ai/v1"`, LangChain sends requests to xAI's servers while keeping the exact same request/response format. There is no separate `langchain-xai` package needed.

**Why the key is named `GROK_API_KEY`:** xAI issues keys with the `xai-` prefix. The env variable is named `GROK_API_KEY` to distinguish it from `OPENAI_API_KEY` even though both are used through `ChatOpenAI`.

In [ ]:
from langchain_openai import ChatOpenAI

grok_llm = ChatOpenAI(
    model="grok-3-mini",
    api_key=GROK_API_KEY,
    base_url="https://api.x.ai/v1",
)

response = grok_llm.invoke("What is LangChain? Answer in one sentence.")
print("xAI Grok →", response.content)

---
## 4. Google — Gemini

**What:** Connects to Google's Gemini models via `ChatGoogleGenerativeAI` from `langchain-google-genai`.

**Why `gemini-2.0-flash`:** Flash is Google's fastest low-latency model — equivalent in role to Claude Haiku or GPT-4o-mini. It is the right choice for quick integration tests.

**How it differs:** Google's package is `langchain-google-genai` (not `langchain-openai`), so it has its own class. However, it still follows the same LangChain `ChatModel` interface — `.invoke()` works identically and returns an `AIMessage`.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

google_llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=GOOGLE_API_KEY,
)

response = google_llm.invoke("What is LangChain? Answer in one sentence.")
print("Google Gemini →", response.content)

---
## 5. Streaming — Claude

**What:** `.stream()` returns a generator that yields `AIMessageChunk` objects one token at a time as the model produces them, instead of waiting for the full response.

**Why use it:** For long responses, streaming lets you display output to the user immediately (word by word) rather than making them wait seconds for the entire reply. Every LangChain `ChatModel` supports `.stream()` with the same interface — no provider-specific code needed.

**How it works:**
- `for chunk in llm.stream(...)` iterates over each token chunk as it arrives.
- `chunk.content` holds the partial text for that chunk (may be empty for the first/last bookend chunks).
- `end=""` and `flush=True` in `print` keep all chunks on the same line in real time.

In [9]:
from langchain_anthropic import ChatAnthropic

stream_llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    anthropic_api_key=ANTHROPIC_API_KEY,
)

print("Anthropic stream →", end=" ")
for chunk in stream_llm.stream("Explain what an LLM is in 3 sentences."):
    print(chunk.content, end="|", flush=True)
print()  # newline after streaming ends

Anthropic stream → |#| Large Language Model (LLM)

An LLM is an artificial intelligence system trained on vast amounts of text data to understand and generate| human language. It works by learning statistical patterns in language, allowing it to predict and produce| coherent text based on input prompts. LLMs like GPT or Claude can perform a| wide variety of language tasks, from answering questions to writing content to explaining concepts.||


---
## 6. Batch — Claude

**What:** `.batch()` accepts a **list** of prompts and sends them all in parallel, returning a list of `AIMessage` objects — one per input.

**Why use it:** When you need to process many independent prompts (e.g. classifying 100 documents), `.batch()` is far faster than calling `.invoke()` in a loop because LangChain dispatches the requests concurrently. The order of the output list matches the order of the input list exactly.

**How it works:**
- Pass a Python list of strings (or message lists) to `.batch()`.
- LangChain runs them concurrently under the hood (using `asyncio` or a thread pool depending on the provider).
- Each element in the returned list is a full `AIMessage` — access `.content` the same way as with `.invoke()`.

In [ ]:
from langchain_anthropic import ChatAnthropic

batch_llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    anthropic_api_key=ANTHROPIC_API_KEY,
)

prompts = [
    "What is LangChain? One sentence.",
    "What is a vector database? One sentence.",
    "What is retrieval-augmented generation (RAG)? One sentence.",
]

responses = batch_llm.batch(prompts)

for prompt, response in zip(prompts, responses):
    print(f"Q: {prompt}")
    print(f"A: {response.content}")
    print()

Q: What is LangChain? One sentence.
A: LangChain is a framework for developing applications powered by large language models (LLMs) that simplifies building complex chains of LLM interactions and integrations with external tools.

Q: What is a vector database? One sentence.
A: A vector database is a specialized database system designed to store and efficiently search high-dimensional numerical vectors, typically used for similarity matching in machine learning and AI applications.

Q: What is retrieval-augmented generation (RAG)? One sentence.
A: Retrieval-augmented generation (RAG) is a technique that enhances AI language models by retrieving relevant external information before generating responses, improving accuracy and reducing hallucinations.



: 